<a href="https://colab.research.google.com/github/atca1977/CLOSDI_TALLER_TUTORIAL_CINTIA_2026/blob/main/CLOSDI_TALLER_TUTORIAL_CINTIA_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01. Montar Google Drive

In [20]:
01. # Importa el módulo de Colab para acceder a Google Drive
from google.colab import drive

# Monta Google Drive en '/content/drive', habilitando el acceso a sus archivos
# desde la maquina virtual de Colab (pide autorización la primera vez)
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 02. Instalar librerías (geemap, eemont)

In [22]:
02. # Instala geemap y eemont (silencioso, sin logs de pip)
!pip install -q geemap eemont

# 03. Autenticación y conexión a Google Earth Engine

In [23]:
03. # Importa el módulo de Earth Engine
import ee

# Autentica la cuenta de Google para usar GEE (pide login la primera vez
# y guarda las credenciales en la VM para el resto de la sesión)
ee.Authenticate()

# Inicializa la sesión de Earth Engine con el proyecto asociado a tu cuenta
ee.Initialize(project='ee-adriancal77') # tu-proyecto-gee

# 04. Área de interés (AOI): mapa interactivo

In [24]:
04. # Importa el módulo de Geemap
import geemap

# Botón y control para recentrar el mapa en Uruguay
import ipywidgets as widgets
from ipyleaflet import WidgetControl

# Crea un mapa en blanco centrado en Uruguay, con herramientas de dibujo habilitadas
Map = geemap.Map(center=[-32.8, -56], zoom=7)
Map.add_basemap('HYBRID')

# Botón que recentra el mapa a la vista inicial de Uruguay
boton_centrar = widgets.Button(icon='home', tooltip='Centrar en Uruguay', layout=widgets.Layout(width='34px', height='34px'))

def centrar_mapa(b):
    Map.center = [-32.8, -56]
    Map.zoom = 7

boton_centrar.on_click(centrar_mapa)
Map.add_control(WidgetControl(widget=boton_centrar, position='topleft'))

Map

Map(center=[-32.8, -56], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

# 05. Parámetros y funciones comunes

In [28]:
# 05. Parámetros y funciones comunes (las usan las celdas 06 a 11)
import os
import eemont
import numpy as np
import pandas as pd
import ipywidgets as widgets
from traitlets import link as jslink
from datetime import datetime
from IPython.display import display

CLOSDI_UMBRAL = 34       # CLOSDI*100 >= umbral -> sombra de nube
UMBRAL_AREA_LIBRE = 10   # % mínimo de área libre para usar un filtro

# Visualización
vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 1500}
vis_closdi = {'min': 0, 'max': 1, 'palette': ["#30123b", "#3e9bfe", "#45f885", "#e0de37", "#f15d13", "#7a0403"]}
vis_ndvi_evi2 = {'min': 0, 'max': 1, 'palette': ["#0000FF", "#8B0000", "#FFFF00", "#90EE90", "#008000", "#004000"]}
vis_veg_100 = {'min': 0, 'max': 100, 'palette': vis_ndvi_evi2['palette']}
vis_closdi_100 = {'min': 0, 'max': 100, 'palette': vis_closdi['palette']}
vis_closdi_bin = {'palette': ['#e0a4c4']}
vis_scl_bin = {'palette': ['#f8a468']}
MAP_WIDTH, MAP_HEIGHT = '420px', '550px'


def clase_scl(img, clases):
    """Máscara 1/0 de los píxeles cuya clase SCL está en `clases`."""
    scl = img.select('SCL')
    m = scl.eq(clases[0])
    for c in clases[1:]:
        m = m.Or(scl.eq(c))
    return m

def calcular_bandas(img, indice='NDVI'):
    """Por escena: índice sin filtro y con filtro (CLOSDI+SCL, solo SCL) + máscaras."""
    s = img.divide(10000)
    N, R = s.select('B8'), s.select('B4')
    if indice == 'NDVI':
        idx = N.subtract(R).divide(N.add(R))
    else:  # EVI2
        idx = N.subtract(R).multiply(2.5).divide(N.multiply(2.4).add(R).add(1))
    closdi = ee.Image().expression(
        '(1.0 - 1.5 * N - 0.1 * R) / (1.0 + 3.5 * N + 4.9 * R)', {'N': N, 'R': R})

    nubes = clase_scl(img, [8, 9, 10])
    sombras = closdi.multiply(100).gte(CLOSDI_UMBRAL)
    total = nubes.Or(sombras)                 # CLOSDI + SCL
    scl_solo = clase_scl(img, [3, 8, 9, 10])  # solo SCL: nubes + sombras de nube

    bandas = ee.Image.cat([
        idx.rename('sin_filtro'),
        idx.updateMask(total.Not()).rename('con_filtro'),
        idx.updateMask(scl_solo.Not()).rename('con_filtro_scl'),
        nubes.rename('nubes'), sombras.rename('sombras'),
        total.rename('total'), scl_solo.rename('scl_solo'),
    ])
    return bandas.set('date_str', img.date().format('ddMMYYYY'),
                      'system:time_start', img.get('system:time_start')).clip(aoi_dibujado_fc)

def por_fecha(indice='NDVI'):
    """Una imagen por día (une pasadas y tiles del mismo día), ordenada por fecha."""
    col = s2_collection.map(lambda img: calcular_bandas(img, indice))
    fechas = col.aggregate_array('date_str').distinct()

    def unir(d):
        mismas = col.filter(ee.Filter.eq('date_str', d))
        return mismas.mosaic().set('date_str', d, 'system:time_start', mismas.first().get('system:time_start'))

    return ee.ImageCollection(fechas.map(unir)).sort('system:time_start')

def tabla_pct(col):
    """DataFrame por fecha: % nubes, sombras, cobertura total y área libre (sin dato = no libre)."""
    kw = dict(geometry=aoi_dibujado_fc.geometry(), scale=10, maxPixels=1e9)

    def stats(img):
        m = img.select(['nubes', 'sombras', 'total']).reduceRegion(ee.Reducer.mean(), **kw)
        libre = (img.select(['total', 'scl_solo', 'nubes']).Not().rename(['libre', 'libre_scl', 'libre_nubes'])
                 .unmask(0).reduceRegion(ee.Reducer.mean(), **kw))
        return ee.Feature(None, m.combine(libre).set('date_str', img.get('date_str')))

    cols = ['date_str', 'nubes', 'sombras', 'total', 'libre', 'libre_scl', 'libre_nubes']
    filas = [f['properties'] for f in col.map(stats).getInfo()['features']]
    df = pd.DataFrame(filas).reindex(columns=cols)
    df[cols[1:]] = df[cols[1:]].astype(float).fillna(0) * 100
    df['FECHA'] = pd.to_datetime(df.date_str, format='%d%m%Y')
    return df.sort_values('FECHA').reset_index(drop=True)

def formato_tabla(df):
    return pd.DataFrame({
        'FECHA': df.FECHA.dt.strftime('%d-%m-%Y'),
        '% NUBES': df.nubes.round(1),
        '% SOMBRAS': df.sombras.round(1),
        '% COBERTURA TOTAL': df.total.round(1),
        '% AREA LIBRE': df.libre.round(1),
    })


def imagen_dia(fecha):
    """Mosaico S2 de un día (une pasadas y tiles del mismo día)."""
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(aoi_dibujado_fc)
           .filterDate(fecha, ee.Date(fecha).advance(1, 'day')))
    return ee.Image(col.mosaic().copyProperties(col.first(), ['system:id', 'system:time_start']))


def indices_eemont(img, nombres):
    """Índices espectrales con eemont (necesita system:id para reconocer el sensor)."""
    esc = ee.Image(img.divide(10000).copyProperties(img, ['system:id', 'system:time_start']))
    return esc.spectralIndices(nombres, online=True).clip(aoi_dibujado_fc)


def panel(titulos, capas, barras=None):
    """Mapas sincronizados + selector de fecha.
    capas(fecha) devuelve, por mapa, una lista de (imagen, vis, nombre).
    barras: {nro_de_mapa: (vis, etiqueta, nombre_de_capa)}."""
    html = lambda t: f"<div style='text-align:center; font-weight:bold; font-size:14px'>{t}</div>"
    mapas = []
    for _ in titulos:
        m = geemap.Map(layout=widgets.Layout(width=MAP_WIDTH, height=MAP_HEIGHT))
        m.add_basemap('HYBRID')
        m.addLayer(aoi_style, {}, 'AOI')
        m.centerObject(aoi_dibujado_fc, 14)
        for ctrl in list(m.controls):
            if type(ctrl).__name__ in ('MapDrawControl', 'FullScreenControl', 'MeasureControl'):
                m.remove_control(ctrl)
        mapas.append(m)

    for a, b in zip(mapas, mapas[1:]):
        jslink((a, 'center'), (b, 'center'))
        jslink((a, 'zoom'), (b, 'zoom'))

    for i, (vis, etiqueta, capa) in (barras or {}).items():
        mapas[i].add_colorbar(vis, label=etiqueta, layer_name=capa, width=2.5, height=0.2)
        mapas[i].controls[-1].widget.layout.width = '200px'

    tit = [widgets.HTML(html(t)) for t in titulos]
    fecha_dd = widgets.Dropdown(options=[(datetime.strptime(d, '%Y-%m-%d').strftime('%d-%m-%Y'), d) for d in dates],
                              value=dates[0], description='FECHA:')
    boton = widgets.Button(description='Centrar en la chacra', icon='crosshairs', button_style='info',
                           layout=widgets.Layout(width='200px'))
    boton.on_click(lambda _: [m.centerObject(aoi_dibujado_fc, 14) for m in mapas])

    def actualizar(fecha):
        for m, lista in zip(mapas, capas(fecha)):
            for nombre in list(m.ee_layers.keys()):
                if nombre != 'AOI':
                    m.remove_ee_layer(nombre)
            for img, vis, nombre in lista:
                m.addLayer(img.clip(aoi_dibujado_fc), vis, nombre)
        fecha_txt = datetime.strptime(fecha, '%Y-%m-%d').strftime('%d-%m-%Y')
        tit[0].value = html(f'{titulos[0]} — {fecha_txt}')

    fecha_dd.observe(lambda c: actualizar(c['new']), names='value')
    actualizar(dates[0])

    caja = widgets.Layout(width=MAP_WIDTH, margin='0 8px 0 0')
    display(widgets.HBox([fecha_dd, boton]),
            widgets.HBox([widgets.VBox([t, m], layout=caja) for t, m in zip(tit, mapas)]))

# Carpeta local de Colab (no Drive) para los archivos de descarga
LOCAL_DIR = '/content/tmp_stack'
os.makedirs(LOCAL_DIR, exist_ok=True)

# 06. Filtrar la colección Sentinel-2 por AOI, fecha y nubes

In [34]:
# 06. Define rango de fechas, arma la colección S2 y muestra la tabla de % nubes/sombras

start_date = '2024-12-01'
end_date = '2025-02-01'

# Recupera el último polígono dibujado en el mapa como ee.Geometry
aoi_dibujado = Map.user_roi

# Lo convierte en FeatureCollection, igual formato que el AOI cargado del asset
aoi_dibujado_fc = ee.FeatureCollection(aoi_dibujado)

# Contorno anaranjado sin relleno, para dibujar sobre el mapa
aoi_style = aoi_dibujado_fc.style(color='orange', fillColor='00000000', width=4)

# Filtra la colección Sentinel-2 SR por AOI, fecha y cobertura de nubes
s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi_dibujado_fc)
    .filterDate(start_date, end_date)
    .sort('system:time_start')
)

# Fechas disponibles, sin duplicados
dates = (
    s2_collection.aggregate_array('system:time_start')
    .map(lambda t: ee.Date(t).format('YYYY-MM-dd'))
    .distinct()
    .sort()
    .getInfo()
)

# % de nubes, sombras, cobertura total y área libre por fecha
formato_tabla(tabla_pct(por_fecha()))


,FECHA,% NUBES,% SOMBRAS,% COBERTURA TOTAL,% AREA LIBRE
0,03-12-2024,0.0,56.8,56.8,43.2
1,08-12-2024,0.0,68.6,68.6,31.4
2,13-12-2024,100.0,0.0,100.0,0.0
3,18-12-2024,0.0,0.9,0.9,99.1
4,23-12-2024,0.0,0.6,0.6,99.4
5,28-12-2024,0.0,0.7,0.7,99.3
6,02-01-2025,0.0,0.8,0.8,99.2
7,07-01-2025,100.0,0.0,100.0,0.0
8,12-01-2025,0.0,0.7,0.7,99.3
9,17-01-2025,0.0,1.3,1.3,98.7


# 07. Comparar RGB, vegetación (NDVI/EVI2) y CLOSDI

In [35]:
# 07. Comparar RGB, vegetación (NDVI/EVI2) y CLOSDI

def capas_07(fecha):
    img = imagen_dia(fecha)
    ind = indices_eemont(img, ['NDVI', 'EVI2', 'CLOSDI'])
    return [
        [(img, vis_rgb, 'RGB')],
        [(ind.select('NDVI'), vis_ndvi_evi2, 'NDVI'), (ind.select('EVI2'), vis_ndvi_evi2, 'EVI2')],
        [(ind.select('CLOSDI'), vis_closdi, 'CLOSDI')],
    ]

panel(['COLOR VERDADERO', 'NDVI / EVI2', 'CLOSDI'], capas_07, # NDVI
      barras={1: (vis_veg_100, 'NDVI / EVI2', 'NDVI'), 2: (vis_closdi_100, 'CLOSDI', 'CLOSDI')})


# 08. CLOSDI vs. SCL: detección de sombras

In [36]:
# 08. CLOSDI vs. SCL: detección de sombras

CLOSDI_UMBRAL = 34       # CLOSDI*100 >= umbral -> sombra de nube

def capas_08(fecha):
    img = imagen_dia(fecha)
    closdi = indices_eemont(img, ['CLOSDI']).select('CLOSDI')
    sombra_closdi = closdi.multiply(100).gte(CLOSDI_UMBRAL).selfMask()
    sombra_scl = clase_scl(img, [2, 3]).selfMask()
    return [
        [(img, vis_rgb, 'RGB')],
        [(img, vis_rgb, 'RGB'), (closdi, vis_closdi, 'CLOSDI'), (sombra_closdi, vis_closdi_bin, 'MASCARA SOMBRAS')],
        [(img, vis_rgb, 'RGB'), (sombra_scl, vis_scl_bin, 'MASCARA SOMBRAS')],
    ]

panel(['COLOR VERDADERO', f'CLOSDI ≥ {CLOSDI_UMBRAL} (SOMBRAS)', 'SENTINEL-2 BANDA SCL (SOMBRAS)'], capas_08,
      barras={1: (vis_closdi_100, 'CLOSDI', 'CLOSDI')})


# 09. CLOSDI + SCL: sombras y nubes combinadas

In [10]:
# 09. CLOSDI + SCL: sombras y nubes combinadas

def capas_09(fecha):
    img = imagen_dia(fecha)
    closdi = indices_eemont(img, ['CLOSDI']).select('CLOSDI')
    nubes = clase_scl(img, [8, 9, 10])
    mascara_closdi = closdi.multiply(100).gte(CLOSDI_UMBRAL).Or(nubes).selfMask()
    mascara_scl = clase_scl(img, [2, 3, 8, 9, 10]).selfMask()
    return [
        [(img, vis_rgb, 'RGB')],
        [(img, vis_rgb, 'RGB'), (closdi, vis_closdi, 'CLOSDI'), (mascara_closdi, vis_closdi_bin, 'NUBES Y SOMBRAS')],
        [(img, vis_rgb, 'RGB'), (mascara_scl, vis_scl_bin, 'NUBES Y SOMBRAS')],
    ]

panel(['COLOR VERDADERO', f'CLOSDI ≥ {CLOSDI_UMBRAL} (SOMBRAS) + SCL (NUBES)', 'SENTINEL-2 BANDA SCL (SOMBRAS + NUBES)'], capas_09,
      barras={1: (vis_closdi_100, 'CLOSDI', 'CLOSDI')})


# 10. Armar y descargar stack de indice espectral

In [11]:
# 10. Tabla de % nubes/sombras + armar y descargar stack (índice en 0-100, solo local, sin Drive)
import os
import zipfile
import rasterio
from google.colab import files

dd_stack = widgets.Dropdown(options=['NDVI', 'EVI2'], value='NDVI', description='INDICE:')
filtro_toggle = widgets.ToggleButtons(options=[('CON FILTRO', True), ('SIN FILTRO', False)], description='FILTRADO:')
boton_tabla = widgets.Button(description='VER TABLA', button_style='info')
boton_stack = widgets.Button(description='DESCARGAR', button_style='success')
salida_stack = widgets.Output()


def ver_tabla(_):
    with salida_stack:
        salida_stack.clear_output()
        print('Calculando tabla...')
        tabla = formato_tabla(tabla_pct(por_fecha()))
        salida_stack.clear_output()
        display(tabla)


def descargar_stack(_):
    indice, filtrar = dd_stack.value, filtro_toggle.value
    sufijo = 'FILTRADO' if filtrar else 'SIN_FILTRAR'
    solo_nubes = indice == 'NDVI'  # NDVI: solo nubes; EVI2: nubes + sombras (CLOSDI + SCL)
    with salida_stack:
        salida_stack.clear_output()
        print(f'Descargando {indice} ({"con" if filtrar else "sin"} filtro'
              f'{" de nubes" if filtrar and solo_nubes else ""})...')

        col = por_fecha(indice)
        df = tabla_pct(col)
        df['libre'] = df.libre_nubes if solo_nubes else df.libre  # área libre según el filtro aplicado

        # Con filtro, las fechas con área libre menor al umbral quedan sin dato
        sin_dato = set(df.date_str[df.libre < UMBRAL_AREA_LIBRE]) if filtrar else set()
        vacias = ee.List(sorted(sin_dato))

        def filtrada(img):
            if not filtrar:
                return img.select('sin_filtro')
            if solo_nubes:
                return img.select('sin_filtro').updateMask(img.select('nubes').Not())
            return img.select('con_filtro')

        def una_banda(img):
            d = ee.String(img.get('date_str'))
            bnd = ee.Image(ee.Algorithms.If(vacias.contains(d), filtrada(img).updateMask(ee.Image(0)), filtrada(img)))
            return bnd.multiply(100).rename([d])  # índice escalado de 0-1 a 0-100

        stack = col.map(una_banda).toBands()
        stack = stack.rename(stack.bandNames().map(lambda n: ee.String(n).split('_').get(-1)))

        periodo = '_'.join(datetime.strptime(x, '%Y-%m-%d').strftime('%d%m%Y') for x in (start_date, end_date))
        base = f'{LOCAL_DIR}/{indice}_{sufijo}_{periodo}'
        NODATA = -9999

        geemap.ee_export_image(stack.unmask(NODATA), filename=f'{base}_STACK.tif', scale=10,
                               region=aoi_dibujado_fc.geometry(), file_per_band=False)

        nombres = stack.bandNames().getInfo()
        with rasterio.open(f'{base}_STACK.tif', 'r+') as src:
            src.nodata = NODATA
            for i, n in enumerate(nombres, start=1):
                src.set_band_description(i, f'{n[:2]}-{n[2:4]}-{n[4:]}')

        t = df.set_index('date_str').loc[nombres]
        pd.DataFrame({
            'BANDA': [f'Band_{i}' for i in range(1, len(nombres) + 1)],
            'FECHA': t.FECHA.dt.strftime('%d-%m-%Y').values,
            'POR_NUBES': t.nubes.round(1).values,
            'POR_SOMBRAS': t.sombras.round(1).values,
            'POR_COBERTURA_TOTAL': t.total.round(1).values,
            'POR_AREA_LIBRE': t.libre.round(1).values,
        }).to_csv(f'{base}.csv', index=False)

        with zipfile.ZipFile(f'{base}.zip', 'w') as zf:
            for ext in ('_STACK.tif', '.csv'):
                zf.write(base + ext, arcname=os.path.basename(base + ext))

        print(f'Listo. ZIP guardado en: {base}.zip')
        files.download(f'{base}.zip')


boton_tabla.on_click(ver_tabla)
boton_stack.on_click(descargar_stack)
display(widgets.VBox([dd_stack, filtro_toggle, widgets.HBox([boton_tabla, boton_stack]), salida_stack]))

# 11. Serie temporal de NDVI / EVI2 filtrada

In [12]:
# 11. Serie temporal de NDVI / EVI2
# NDVI: sin filtro vs. filtro de nubes (SCL). EVI2: sin filtro, solo SCL (nubes + sombras) y CLOSDI + SCL
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from google.colab import files

dd_serie = widgets.Dropdown(options=['NDVI', 'EVI2'], value='NDVI', description='INDICE:')
boton_graficar = widgets.Button(description='GRAFICAR', button_style='info')
boton_serie = widgets.Button(description='DESCARGAR SERIE', button_style='success')
salida_serie = widgets.Output()
serie_actual = {}  # última serie calculada, para no recalcularla al descargar

SERIES_EVI2 = [  # columna, etiqueta, estilo de línea, color
    ('sin_filtro', 'Sin filtro', '--', '#B0B0B0'),
    ('con_filtro_scl', 'Con filtro (solo SCL)', ':', '#C77B00'),
    ('con_filtro', 'Con filtro (CLOSDI + SCL)', '-', '#2C5F2D'),
]
SERIES_NDVI = [
    ('sin_filtro', 'Sin filtro', '--', '#B0B0B0'),
    ('con_filtro', 'Con filtro (solo nubes, SCL)', '-', '#2C5F2D'),
]
# Nombres de columna del CSV, por índice (mismo campo 'con_filtro' se llama distinto según el caso)
NOMBRES_CSV_NDVI = {'sin_filtro': 'SIN_FILTRO', 'con_filtro': 'CON_FILTRO_NUBES'}
NOMBRES_CSV_EVI2 = {'sin_filtro': 'SIN_FILTRO', 'con_filtro_scl': 'CON_FILTRO_SCL', 'con_filtro': 'CON_FILTRO_CLOSDI_Y_SCL'}


def graficar(_):
    indice = dd_serie.value
    solo_nubes = indice == 'NDVI'  # NDVI: las sombras no lo afectan, se filtran solo nubes
    series = SERIES_NDVI if solo_nubes else SERIES_EVI2
    with salida_serie:
        salida_serie.clear_output()
        print(f'Calculando serie temporal de {indice}...')

        col = por_fecha(indice)
        cols = [s[0] for s in series]

        def mediana(img):
            if solo_nubes:  # reemplaza 'con_filtro' por el índice enmascarado solo con nubes
                con = img.select('sin_filtro').updateMask(img.select('nubes').Not()).rename('con_filtro')
                img = img.addBands(con, overwrite=True)
            v = img.select(cols).reduceRegion(reducer=ee.Reducer.median(), geometry=aoi_dibujado_fc.geometry(),
                                              scale=10, maxPixels=1e9)
            return ee.Feature(None, v.set('date_str', img.get('date_str')))

        filas = [f['properties'] for f in col.map(mediana).getInfo()['features']]
        med = pd.DataFrame(filas).reindex(columns=['date_str'] + cols)
        med[cols] = med[cols].astype(float) * 100  # fechas 100% enmascaradas quedan en NaN

        df = tabla_pct(col).merge(med, on='date_str', how='left')

        # Si el área libre del filtro es menor al umbral, esa serie queda sin dato
        df['libre_filtro'] = df.libre_nubes if solo_nubes else df.libre
        df.loc[df.libre_filtro < UMBRAL_AREA_LIBRE, 'con_filtro'] = np.nan
        if not solo_nubes:
            df.loc[df.libre_scl < UMBRAL_AREA_LIBRE, 'con_filtro_scl'] = np.nan

        serie_actual.update(indice=indice, df=df, series=series, solo_nubes=solo_nubes)

        x = df.FECHA.dt.strftime('%d-%m-%Y')
        plt.figure(figsize=(10, 4))
        for c, etiqueta, estilo, color in series:
            plt.plot(x, df[c], marker='o', linestyle=estilo, color=color, label=etiqueta)
        plt.xticks(rotation=45, ha='right')
        plt.ylim(0, 100)
        plt.gca().yaxis.set_major_locator(mticker.MultipleLocator(10))
        plt.ylabel(f'{indice} (x100)')
        plt.title(f'Serie temporal de {indice} — parcela de interés')
        plt.legend()
        plt.tight_layout()
        plt.show()


def descargar_serie(_):
    with salida_serie:
        if not serie_actual:
            print('Primero generá el gráfico con el botón GRAFICAR.')
            return

        indice, df, series = serie_actual['indice'], serie_actual['df'], serie_actual['series']
        nombres_csv = NOMBRES_CSV_NDVI if serie_actual['solo_nubes'] else NOMBRES_CSV_EVI2
        ruta = f'{LOCAL_DIR}/{indice}_serie_temporal_con_sin_filtro.csv'

        out = {'FECHA': df.FECHA.dt.strftime('%d-%m-%Y')}
        for c, *_ in series:
            out[f'{indice}_{nombres_csv[c]}'] = df[c].round(1)
        pd.DataFrame(out).to_csv(ruta, index=False)

        files.download(ruta)


boton_graficar.on_click(graficar)
boton_serie.on_click(descargar_serie)
display(widgets.VBox([dd_serie, widgets.HBox([boton_graficar, boton_serie]), salida_serie]))